# 03 · Full CascadeFlow VLM Experiment

This notebook runs a **complete CascadeFlow experiment** on your VLM dataset,
with images and proper evaluation, so you can later compare it against your router.

It does the following:

1. Load the **test dataset** (with `sample_id`, prompt, ground truth, and image path).
2. Configure **model pricing** and VLM endpoints (vLLM).
3. Build a **CascadeFlow agent** over your VLM pool, sorted by cost.
4. For each sample:
   - Send the **same prompt + image** to CascadeFlow,
   - Get the final model's answer,
   - Evaluate it with your `Scorer` from `evaluation.py`,
   - Compute true dollar **cost** from token counts,
   - Record latency and routing metadata.
5. Save per-sample results as a Parquet file:
   `outputs/cascadeflow/cascadeflow_results.parquet`.
6. Save a summary JSON with global metrics and per-task stats.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import time
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional

# NOTE: Avoid applying nest_asyncio here — httpx/anyio expect a normal asyncio loop,
# and patching the loop causes "unknown async library" errors.
# Jupyter already supports top-level await, so no extra loop patching is required.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from cascadeflow import CascadeAgent, ModelConfig
from evaluation import Scorer


In [3]:

DATA_ROOT = Path.cwd().parent.parent / "dataset"

# Path to your final test dataset with images
# Expecting columns like: sample_id, prompt_raw, ground_truth, ground_truth_type,
# router_task, source_config, image_path, etc.
TEST_DATASET_PATH = DATA_ROOT / "final_dataset" / "router_lexico" / "router_test_trainer.parquet"

OUTPUT_DIR = Path("outputs/cascadeflow")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_DATASET_PATH, OUTPUT_DIR


(PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset/router_lexico/router_test_trainer.parquet'),
 PosixPath('outputs/cascadeflow'))

## 1 · Configure model pool, pricing, and endpoints

We define a **shared cost model** for all VLMs, and tell CascadeFlow how to reach them via vLLM.
Edit the pricing numbers and base URLs to match your cluster.


In [4]:

# vLLM endpoints for each model (EDIT ports / URLs)
MODEL_ENDPOINTS: Dict[str, str] = {
    "Qwen/Qwen2.5-VL-3B-Instruct":   "http://localhost:8801/v1",
    "Qwen/Qwen2.5-VL-7B-Instruct":   "http://localhost:8802/v1",
    "google/gemma-3-27b-it":         "http://localhost:8803/v1",
    "Qwen/Qwen3-VL-8B-Thinking":     "http://localhost:8804/v1",
    "deepseek-ai/DeepSeek-OCR":      "http://localhost:8805/v1",
}

# Model
# pricing (EDIT these numbers to match your assumptions)
# Prices are in USD per 1k tokens.
MODEL_PRICING: Dict[str, Dict[str, float]] = {
    "Qwen/Qwen2.5-VL-3B-Instruct":   {"prompt_per_1k": 0.0002, "completion_per_1k": 0.0002},
    "Qwen/Qwen2.5-VL-7B-Instruct":   {"prompt_per_1k": 0.0003, "completion_per_1k": 0.0003},
    "google/gemma-3-27b-it":         {"prompt_per_1k": 0.0008, "completion_per_1k": 0.0008},
    "Qwen/Qwen3-VL-8B-Thinking":     {"prompt_per_1k": 0.0010, "completion_per_1k": 0.0010},
    "deepseek-ai/DeepSeek-OCR":      {"prompt_per_1k": 0.0004, "completion_per_1k": 0.0004},
}

def compute_true_cost(model_name: str, prompt_tokens: int, completion_tokens: int) -> float:
    pricing = MODEL_PRICING.get(model_name)
    if pricing is None:
        return 0.0
    return (
        (prompt_tokens / 1000.0) * pricing["prompt_per_1k"]
        + (completion_tokens / 1000.0) * pricing["completion_per_1k"]
    )



### 1.1 · Health check vLLM endpoints

We ping `/models` on each endpoint to confirm it is alive and serving the expected model.


In [5]:
def check_vllm_health(base_url: str) -> Dict[str, Any]:
    info: Dict[str, Any] = {"base_url": base_url, "ok": False, "model_ids": [], "error": None}
    try:
        resp = requests.get(base_url.rstrip("/") + "/models", timeout=5)
        resp.raise_for_status()
        data = resp.json()
        ids = [m.get("id") for m in data.get("data", [])]
        info["ok"] = True
        info["model_ids"] = ids
    except Exception as e:
        info["error"] = str(e)
    return info

health_status: Dict[str, Dict[str, Any]] = {}
for model_name, url in MODEL_ENDPOINTS.items():
    info = check_vllm_health(url)
    health_status[model_name] = info
    print(f"{model_name}: ok={info['ok']}, ids={info['model_ids']}, error={info['error']}")


Qwen/Qwen2.5-VL-3B-Instruct: ok=True, ids=['Qwen/Qwen3-VL-8B-Thinking'], error=None
Qwen/Qwen2.5-VL-7B-Instruct: ok=True, ids=['Qwen/Qwen2.5-VL-7B-Instruct'], error=None
google/gemma-3-27b-it: ok=True, ids=['Qwen/Qwen2.5-VL-3B-Instruct'], error=None
Qwen/Qwen3-VL-8B-Thinking: ok=True, ids=['deepseek-ai/DeepSeek-OCR'], error=None
deepseek-ai/DeepSeek-OCR: ok=True, ids=['PatronusAI/glider'], error=None


### 1.2 · Build CascadeFlow model configs

We keep only healthy models, then sort them **by cost** from cheapest to most expensive.
This defines the cascade order: Tier 1 = cheapest, last tier = most expensive fallback.


In [6]:
available_models = [
    m for m, info in health_status.items()
    if info["ok"] and m in MODEL_PRICING
]

if not available_models:
    raise RuntimeError("No healthy models found. Fix MODEL_ENDPOINTS or vLLM servers.")

# Sort by cost (prompt cost as proxy)
available_models = sorted(
    available_models,
    key=lambda m: MODEL_PRICING[m]["prompt_per_1k"],
)

print("Cascade order (cheapest -> most expensive):")
for m in available_models:
    print("  -", m, "price:", MODEL_PRICING[m])

cascade_models: List[ModelConfig] = []
for m in available_models:
    info = health_status[m]
    base_url = info["base_url"]
    model_id = info["model_ids"][0] if info["model_ids"] else m

    cfg = ModelConfig(
        name=model_id,
        provider="vllm",
        base_url=base_url,
        cost=MODEL_PRICING[m]["prompt_per_1k"],  # relative cost for CascadeFlow
        quality_threshold=0.75,                  # EDIT thresholds if you like
        metadata={"logical_name": m},
    )
    cascade_models.append(cfg)

cascade_models


Cascade order (cheapest -> most expensive):
  - Qwen/Qwen2.5-VL-3B-Instruct price: {'prompt_per_1k': 0.0002, 'completion_per_1k': 0.0002}
  - Qwen/Qwen2.5-VL-7B-Instruct price: {'prompt_per_1k': 0.0003, 'completion_per_1k': 0.0003}
  - deepseek-ai/DeepSeek-OCR price: {'prompt_per_1k': 0.0004, 'completion_per_1k': 0.0004}
  - google/gemma-3-27b-it price: {'prompt_per_1k': 0.0008, 'completion_per_1k': 0.0008}
  - Qwen/Qwen3-VL-8B-Thinking price: {'prompt_per_1k': 0.001, 'completion_per_1k': 0.001}


[ModelConfig(name='Qwen/Qwen3-VL-8B-Thinking', provider='vllm', cost=0.0002, keywords=[], domains=[], max_tokens=4096, system_prompt=None, temperature=0.7, api_key=None, base_url='http://localhost:8801/v1', extra={}, speed_ms=1000, quality_score=0.7, supports_tools=True, quality_threshold=0.75, metadata={'logical_name': 'Qwen/Qwen2.5-VL-3B-Instruct'}),
 ModelConfig(name='Qwen/Qwen2.5-VL-7B-Instruct', provider='vllm', cost=0.0003, keywords=[], domains=[], max_tokens=4096, system_prompt=None, temperature=0.7, api_key=None, base_url='http://localhost:8802/v1', extra={}, speed_ms=1000, quality_score=0.7, supports_tools=True, quality_threshold=0.75, metadata={'logical_name': 'Qwen/Qwen2.5-VL-7B-Instruct'}),
 ModelConfig(name='PatronusAI/glider', provider='vllm', cost=0.0004, keywords=[], domains=[], max_tokens=4096, system_prompt=None, temperature=0.7, api_key=None, base_url='http://localhost:8805/v1', extra={}, speed_ms=1000, quality_score=0.7, supports_tools=True, quality_threshold=0.75, 

### 1.3 · Create CascadeAgent


In [7]:
agent = CascadeAgent(models=cascade_models)
agent


LiteLLM not installed. Cost tracking will use fallback estimates. Install with: pip install litellm
LiteLLM not available. Cost calculations will use fallback estimates.
LiteLLM not available. Cost calculations will use fallback estimates.
LiteLLM not available. Cost calculations will use fallback estimates.
LiteLLM not available. Cost calculations will use fallback estimates.
LiteLLM not available. Cost calculations will use fallback estimates.
FastEmbed not available. Install with: pip install fastembed


## 2 · Load test dataset

We load your **final test dataset**. It should include at least:

- `sample_id`
- `prompt_raw`
- `ground_truth`
- `ground_truth_type` (e.g. `mc`, `numeric`, `freeform`, `exact`)
- `router_task`
- `source_config`
- `image_path` (path to the main image file)


In [8]:
assert TEST_DATASET_PATH.exists(), f"Test dataset not found: {TEST_DATASET_PATH}"
test_df = pd.read_parquet(TEST_DATASET_PATH)

print("Test shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())
print(test_df[["sample_id", "router_task", "source_config"]].head())


Test shape: (13707, 33)
Columns: ['sample_id', 'image_path', 'image_bytes_hash', 'prompt_raw', 'img_width', 'img_height', 'img_aspect_ratio', 'txt_prompt_length_chars', 'txt_prompt_length_words', 'router_task', 'source_dataset', 'source_config', 'txt_question_type', 'txt_has_mc_options', 'ground_truth', 'ground_truth_type', 'router_best_model_id', 'router_best_model_name', 'router_chosen_perf', 'router_chosen_cost', 'image_png', 'input_text', 'label_id', 'label_name', 'utility_scheme', 'hier_w_sample', 'hier_w_task', 'hier_w_global', 'router_soft_p_deepseek_ocr', 'router_soft_p_qwen2_5_vl_3b', 'router_soft_p_qwen2_5_vl_7b', 'router_soft_p_qwen3_vl_8b_thinking', 'router_soft_p_gemma_3_27b']
                     sample_id        router_task source_config
0  ai2d_00004_bf3d9c5fd30bf304  diagram_reasoning          ai2d
1  ai2d_00007_e6f58451a22503d1  diagram_reasoning          ai2d
2  ai2d_00019_e5f934cfc0951972  diagram_reasoning          ai2d
3  ai2d_00021_59cfa73bc8485386  diagram_reaso

In [9]:
# Optional: subsample for quicker runs during debugging
MAX_SAMPLES = 10  # e.g., 500 for a quick pass; set to None for full test
RANDOM_SEED = 42

if MAX_SAMPLES is not None and len(test_df) > MAX_SAMPLES:
    test_df = test_df.sample(n=MAX_SAMPLES, random_state=RANDOM_SEED).reset_index(drop=True)

len(test_df)


10

In [10]:
def build_vlm_messages(prompt: str, image_path: Optional[str]) -> List[Dict[str, Any]]:
    content: List[Dict[str, Any]] = []
    content.append({"type": "text", "text": prompt})

    if image_path is not None and isinstance(image_path, str) and image_path.strip():
        # Here we use a file:// URL convention; adjust if needed.
        image_url = f"file://{image_path}"
        content.append({"type": "image", "image_url": image_url})

    return [{"role": "user", "content": content}]



In [11]:
@dataclass
class CascadeEvalResult:
    sample_id: str
    source_config: str
    router_task: str

    prompt_raw: str
    ground_truth: str
    ground_truth_type: str
    image_path: Optional[str]

    cascade_model: str
    cascade_logical_model: str
    cascaded: bool
    draft_accepted: Optional[bool]

    response_raw: str

    prompt_tokens: int
    completion_tokens: int
    cascade_cost: float
    cascade_latency_ms: float

    # Scoring
    is_correct: bool
    score_f1: float
    score_exact_match: float
    score_exact_match_normalized: float
    score_contains_gt: float
    score_gt_in_response: float
    score_numeric_match: Optional[float]
    score_mc_letter_match: Optional[float]
    pred_answer_letter: Optional[str]
    gt_answer_letter: Optional[str]

    # Error tracking
    error: Optional[str] = None


In [12]:
async def evaluate_sample(row: pd.Series, agent: CascadeAgent) -> CascadeEvalResult:
    sample_id = str(row.get("sample_id", ""))
    router_task = str(row.get("router_task", "unknown"))
    source_config = str(row.get("source_config", "unknown"))
    prompt_raw = str(row.get("prompt_raw", ""))
    ground_truth = str(row.get("ground_truth", ""))
    gt_type = str(row.get("ground_truth_type", "exact")) or "exact"
    image_path = row.get("image_path", None)

    messages = build_vlm_messages(prompt_raw, image_path)
    print(messages)
    try:
        t0 = time.time()
        result = await agent.run(query=prompt_raw, max_tokens=512, temperature=0.0)
        t1 = time.time()

        response = getattr(result, "content", "") or ""
        cascade_latency_ms = getattr(result, "latency_ms", (t1 - t0) * 1000.0)
        cascaded = getattr(result, "cascaded", False)
        draft_accepted = getattr(result, "draft_accepted", None)
        used_model = getattr(result, "model_used", "") or ""

        md = getattr(result, "metadata", {}) or {}
        prompt_tokens = int(md.get("prompt_tokens", md.get("input_tokens", 0)) or 0)
        completion_tokens = int(md.get("completion_tokens", md.get("output_tokens", 0)) or 0)

        logical_name: str = used_model
        for cfg in agent.models:
            if cfg.name == used_model:
                logical_name = (cfg.metadata or {}).get("logical_name", used_model)
                break
        
        cascade_cost = compute_true_cost(logical_name, prompt_tokens, completion_tokens)

        scores = Scorer.compute_all_scores(pred=response, gt=ground_truth, gt_type=gt_type)

        print(
        f"    ↪ CascadeFlow routed to model={logical_name} | "
        f"cascaded={cascaded}, draft_accepted={draft_accepted}, "
        f"tokens=({prompt_tokens}, {completion_tokens}), cost={cascade_cost:.6f}"
)

        return CascadeEvalResult(
            sample_id=sample_id,
            source_config=source_config,
            router_task=router_task,
            prompt_raw=prompt_raw,
            ground_truth=ground_truth,
            ground_truth_type=gt_type,
            image_path=image_path if isinstance(image_path, str) else None,
            cascade_model=used_model,
            cascade_logical_model=logical_name,
            cascaded=cascaded,
            draft_accepted=draft_accepted,
            response_raw=response,
            prompt_tokens=prompt_tokens,
            completion_tokens=completion_tokens,
            cascade_cost=cascade_cost,
            cascade_latency_ms=cascade_latency_ms,
            is_correct=bool(scores.get("is_correct", False)),
            score_f1=float(scores.get("score_f1", 0.0) or 0.0),
            score_exact_match=float(scores.get("score_exact_match", 0.0) or 0.0),
            score_exact_match_normalized=float(scores.get("score_exact_match_normalized", 0.0) or 0.0),
            score_contains_gt=float(scores.get("score_contains_gt", 0.0) or 0.0),
            score_gt_in_response=float(scores.get("score_gt_in_response", 0.0) or 0.0),
            score_numeric_match=scores.get("score_numeric_match"),
            score_mc_letter_match=scores.get("score_mc_letter_match"),
            pred_answer_letter=scores.get("pred_answer_letter"),
            gt_answer_letter=scores.get("gt_answer_letter"),
            error=None,
        )

    except Exception as e:
        return CascadeEvalResult(
            sample_id=sample_id,
            source_config=source_config,
            router_task=router_task,
            prompt_raw=prompt_raw,
            ground_truth=ground_truth,
            ground_truth_type=gt_type,
            image_path=image_path if isinstance(image_path, str) else None,
            cascade_model="",
            cascade_logical_model="",
            cascaded=False,
            draft_accepted=None,
            response_raw="",
            prompt_tokens=0,
            completion_tokens=0,
            cascade_cost=0.0,
            cascade_latency_ms=0.0,
            is_correct=False,
            score_f1=0.0,
            score_exact_match=0.0,
            score_exact_match_normalized=0.0,
            score_contains_gt=0.0,
            score_gt_in_response=0.0,
            score_numeric_match=None,
            score_mc_letter_match=None,
            pred_answer_letter=None,
            gt_answer_letter=None,
            error=str(e),
        )


## 3 · Helper: build VLM messages with text + image

We construct OpenAI-style messages that include both the text prompt and the image.
Adapt this to match the API your vLLM endpoints expect (e.g., `image_url` vs `image` bytes).


In [13]:
from typing import Any

def build_vlm_messages(prompt: str, image_path: Optional[str]) -> List[Dict[str, Any]]:
    content: List[Dict[str, Any]] = []
    content.append({"type": "text", "text": prompt})

    if image_path is not None and isinstance(image_path, str) and image_path.strip():
        # Here we use a file:// URL convention; adjust if needed.
        image_url = f"file://{image_path}"
        content.append({"type": "image", "image_url": image_url})

    return [{"role": "user", "content": content}]


## 4 · Evaluation loop

We define a dataclass to store per-sample results, then an async `evaluate_sample` function to:

1. Build messages with prompt + image.
2. Call `agent.run(...)`.
3. Extract final model, tokens, cost, latency, and routing flags.
4. Evaluate using `Scorer.compute_all_scores`.


### 4.1 · Run CascadeFlow over all test samples


In [14]:
from dataclasses import dataclass
from typing import Optional
from datetime import datetime
from tqdm.auto import tqdm
import time
import pandas as pd

# ---------------------------------------------------------
# 1. Per-sample result container
# ---------------------------------------------------------
@dataclass
class EvaluationResult:
    sample_id: str
    source_config: str
    router_task: str

    prompt: str
    ground_truth: str

    model_used: str
    response: str
    cascaded: bool
    draft_accepted: bool

    latency_ms: float
    total_cost: float
    total_tokens: int

    is_correct: Optional[bool] = None
    exact_match: Optional[bool] = None

    error: Optional[str] = None
    timestamp: str = ""


# ---------------------------------------------------------
# 2. Single-sample evaluation through CascadeAgent
# ---------------------------------------------------------
async def evaluate_sample(row: pd.Series, agent: CascadeAgent) -> EvaluationResult:
    """Evaluate a single sample through the CascadeFlow agent."""
    sample_id = row["sample_id"]
    source_config = row.get("source_config", "unknown")
    router_task = row.get("router_task", "unknown")
    prompt_raw = row["prompt_raw"]
    ground_truth_raw = str(row.get("ground_truth", ""))

    try:
        t0 = time.time()

        # --- Call CascadeFlow ---
        # NOTE: currently we only send text. If you want to add images later,
        # replace `query=prompt_raw` with `messages=[...]` using your image path.
        result = await agent.run(
            query=prompt_raw,
            max_tokens=512,
            temperature=0.0,  # deterministic for eval
        )

        t1 = time.time()

        # --- Extract fields from CascadeFlow result ---
        total_cost = float(getattr(result, "total_cost", 0.0) or 0.0)
        total_tokens = int(getattr(result, "total_tokens", 0) or 0)
        latency_ms = float(getattr(result, "latency_ms", (t1 - t0) * 1000.0) or 0.0)
        cascaded = bool(getattr(result, "cascaded", False))
        draft_accepted = bool(getattr(result, "draft_accepted", False))
        model_used = getattr(result, "model_used", "unknown") or "unknown"
        response = getattr(result, "content", "") or ""

        # --- Simple accuracy checks ---
        ground_truth = ground_truth_raw.strip().lower()
        response_clean = response.strip().lower()

        exact_match = ground_truth == response_clean if ground_truth else None
        is_correct = (ground_truth in response_clean) if ground_truth else None

        return EvaluationResult(
            sample_id=sample_id,
            source_config=source_config,
            router_task=router_task,
            prompt=prompt_raw[:200],              # truncate for storage
            ground_truth=ground_truth_raw,
            model_used=model_used,
            response=response[:500],              # truncate for storage
            cascaded=cascaded,
            draft_accepted=draft_accepted,
            latency_ms=latency_ms,
            total_cost=total_cost,
            total_tokens=total_tokens,
            is_correct=is_correct,
            exact_match=exact_match,
            error=None,
            timestamp=datetime.now().isoformat(),
        )

    except Exception as e:
        # On failure, record the error but keep the row
        return EvaluationResult(
            sample_id=sample_id,
            source_config=source_config,
            router_task=router_task,
            prompt=prompt_raw[:200],
            ground_truth=ground_truth_raw,
            model_used="error",
            response="",
            cascaded=False,
            draft_accepted=False,
            latency_ms=0.0,
            total_cost=0.0,
            total_tokens=0,
            is_correct=False,
            exact_match=False,
            error=str(e),
            timestamp=datetime.now().isoformat(),
        )



In [15]:
import nest_asyncio
nest_asyncio.apply()
print("Jupyter async patch applied!")


Jupyter async patch applied!


In [16]:

# ---------------------------------------------------------
# 3. Run evaluation over the whole test_df
# ---------------------------------------------------------

evaluation_results: list[EvaluationResult] = []

print(f"Starting cascade evaluation on {len(test_df)} samples...\n")
start_time = time.time()

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Evaluating"):
    # NOTE: this cell must be run in Jupyter (or IPython) where top-level `await` is allowed
    result = await evaluate_sample(row, agent)
    evaluation_results.append(result)

    # Optional: small log every N samples
    if idx % 50 == 0:
        print(
            f"[{idx+1}/{len(test_df)}] sample_id={result.sample_id}, "
            f"model={result.model_used}, cost={result.total_cost:.6f}, "
            f"latency={result.latency_ms:.1f} ms, correct={result.is_correct}"
        )

    # Track costs in your CostTracker (if you have it)
    if result.total_cost and result.model_used != "error":
        cost_tracker.add_cost(
            model=result.model_used,
            provider="vllm",
            tokens=result.total_tokens,
            cost=result.total_cost,
            query_id=result.sample_id,
        )

elapsed_time = time.time() - start_time

print("\n✓ Evaluation complete!")
print(f"  Total time: {elapsed_time:.2f}s")
print(f"  Avg time per sample: {elapsed_time/len(test_df):.2f}s")
print(f"  Samples processed: {len(evaluation_results)}")

successful = [r for r in evaluation_results if r.error is None]
failed = [r for r in evaluation_results if r.error is not None]
print(f"  Successful: {len(successful)}")
print(f"  Failed: {len(failed)}")


Starting cascade evaluation on 10 samples...



Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

VLLM: ✗ Not retrying unknown on attempt 1/3: unknown async library, or not in async context
Drafter error: unknown async library, or not in async context
Traceback (most recent call last):
  File "/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/code_base/cascadeflow/cascadeflow_env/lib/python3.14/site-packages/httpcore/_async/connection_pool.py", line 228, in handle_async_request
    closing = self._assign_requests_to_connections()
  File "/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/code_base/cascadeflow/cascadeflow_env/lib/python3.14/site-packages/httpcore/_async/connection_pool.py", line 328, in _assign_requests_to_connections
    pool_request.assign_to_connection(connection)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/code_base/cascadeflow/cascadeflow_env/lib/python3.14/site-p

[1/10] sample_id=robut_wtq_01038_fc90a68db1a72ace, model=Qwen/Qwen3-VL-8B-Thinking+deepseek-ai/DeepSeek-OCR, cost=0.000000, latency=11.5 ms, correct=False

✓ Evaluation complete!
  Total time: 0.09s
  Avg time per sample: 0.01s
  Samples processed: 10
  Successful: 9
  Failed: 1


## 5 · Save per-sample results


In [ ]:
results_df = pd.DataFrame([asdict(r) for r in evaluation_results])
print("Results shape:", results_df.shape)
results_df.head()

parquet_path = OUTPUT_DIR / "cascadeflow_results.parquet"
results_df.to_parquet(parquet_path, index=False)
print("Saved per-sample results to:", parquet_path)


## 6 · Summary metrics and quick plots


In [ ]:
success_df = results_df[results_df["error"].isna()] if "error" in results_df.columns else results_df.copy()
print("Total samples:", len(results_df))
print("Successful samples:", len(success_df))

overall_accuracy = success_df["is_correct"].mean() if len(success_df) > 0 else 0.0
total_cost = success_df["cascade_cost"].sum()
avg_cost = success_df["cascade_cost"].mean() if len(success_df) > 0 else 0.0
avg_latency = success_df["cascade_latency_ms"].mean() if len(success_df) > 0 else 0.0

print("Overall accuracy:", overall_accuracy)
print("Total cost:", total_cost)
print("Avg cost per sample:", avg_cost)
print("Avg latency (ms):", avg_latency)


In [ ]:
model_stats = success_df.groupby("cascade_logical_model").agg(
    n_samples=("sample_id", "count"),
    avg_cost=("cascade_cost", "mean"),
    total_cost=("cascade_cost", "sum"),
    avg_latency_ms=("cascade_latency_ms", "mean"),
    accuracy=("is_correct", "mean"),
).reset_index()

model_stats


In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(model_stats["cascade_logical_model"], model_stats["avg_cost"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Avg cost per sample (USD)")
plt.title("CascadeFlow: avg cost per model")
plt.tight_layout()
plt.show()


In [ ]:
task_stats = success_df.groupby("router_task").agg(
    n_samples=("sample_id", "count"),
    accuracy=("is_correct", "mean"),
    avg_cost=("cascade_cost", "mean"),
    total_cost=("cascade_cost", "sum"),
    avg_latency_ms=("cascade_latency_ms", "mean"),
).reset_index()

task_stats


In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(task_stats["router_task"], task_stats["accuracy"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Accuracy")
plt.title("CascadeFlow accuracy per task")
plt.tight_layout()
plt.show()


## 7 · Save summary JSON

We save a summary JSON file so the comparison notebook can load it, and so you can quote numbers in your report.


In [ ]:
summary = {
    "n_total": int(len(results_df)),
    "n_success": int(len(success_df)),
    "overall_accuracy": float(overall_accuracy),
    "total_cost": float(total_cost),
    "avg_cost": float(avg_cost),
    "avg_latency_ms": float(avg_latency),
    "per_model": model_stats.to_dict(orient="records"),
    "per_task": task_stats.to_dict(orient="records"),
}

summary_path = OUTPUT_DIR / "cascadeflow_summary.json"
with summary_path.open("w") as f:
    import json
    json.dump(summary, f, indent=2)

summary_path
